In [1]:
from utils import functions
from utils import config
from utils import plotting_numeric
from utils import bigquery_functions
from utils.logger_config import setup_logger

import matplotlib.pyplot as plt
import concurrent.futures
import queue
import pandas as pd
import os

# Set font sizes
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['axes.labelsize'] = 15
plt.rcParams['axes.titlesize'] = 15

# Set font family
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ["Geneva", "Menio", 'Helvetica']

/Users/nurfaldi/datakota/politico/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
query_7_51 = (
   """
   SELECT 
      rtv.kode_kelurahan,
      rtv.kode_kabupaten,
      cd.kelurahan,
      cd.kabupaten,
      SUM(rtv.total_suara_dpt) AS total_suara_dpt,
      SUM(rtv.target_voters) AS target_voters,
      SUM(rtv.no_of_people_reached) AS no_of_people_reached,
      SUM(rtv.house_visit_target) AS house_visit_target,
      SUM(rtv.door_to_door_per_day_target) AS door_to_door_per_day_target,
      SUM(rtv.volunteer_needs) AS volunteer_needs,

   FROM `personal-nurfaldi.datakota.sultra_dpt_rtv_weighted_calculated_7_51_bq` rtv
   LEFT JOIN `personal-nurfaldi.datakota.sultra_code_mapping` cd ON cd.kode_kelurahan = rtv.kode_kelurahan
   GROUP BY kode_kelurahan, kode_kabupaten, cd.kelurahan, cd.kabupaten

   """
)

query_7_35 = (
   """
   SELECT 
      rtv.kode_kelurahan,
      rtv.kode_kabupaten,
      cd.kelurahan,
      cd.kabupaten,
      SUM(rtv.total_suara_dpt) AS total_suara_dpt,
      SUM(rtv.target_voters) AS target_voters,
      SUM(rtv.no_of_people_reached) AS no_of_people_reached,
      SUM(rtv.house_visit_target) AS house_visit_target,
      SUM(rtv.door_to_door_per_day_target) AS door_to_door_per_day_target,
      SUM(rtv.volunteer_needs) AS volunteer_needs,

   FROM `personal-nurfaldi.datakota.sultra_dpt_rtv_weighted_calculated_7_35_bq` rtv
   LEFT JOIN `personal-nurfaldi.datakota.sultra_code_mapping` cd ON cd.kode_kelurahan = rtv.kode_kelurahan
   GROUP BY kode_kelurahan, kode_kabupaten, cd.kelurahan, cd.kabupaten

   """
)

df_sultra_kel_rtv_weighted_calculated_7_51_bq = bigquery_functions.query_bq(query_7_51)
df_sultra_kel_rtv_weighted_calculated_7_35_bq = bigquery_functions.query_bq(query_7_35)

/Users/nurfaldi/datakota/politico/venv/lib/python3.9/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [3]:
# Rename Columns for plotting
df_sultra_kel_rtv_weighted_calculated_7_51_bq.rename(columns={
    'total_suara_dpt': 'Total Suara DPT',
    'target_voters': 'Target Suara 51%',
    'no_of_people_reached': 'Target Orang yang Ditemui 51%',
    'house_visit_target': 'Target Visit Rumah 51%',
    'door_to_door_per_day_target': 'Target D2D Perhari 51%',
    'volunteer_needs': 'Kebutuhan Relawan 51%'
}, inplace=True)

df_sultra_kel_rtv_weighted_calculated_7_35_bq.rename(columns={
    'total_suara_dpt': 'Total Suara DPT',
    'target_voters': 'Target Suara 35%',
    'no_of_people_reached': 'Target Orang yang Ditemui 35%',
    'house_visit_target': 'Target Visit Rumah 35%',
    'door_to_door_per_day_target': 'Target D2D Perhari 35%',
    'volunteer_needs': 'Kebutuhan Relawan 35%'
}, inplace=True)

# Extract kode_kelurahan from data
df_merge_35_51 = df_sultra_kel_rtv_weighted_calculated_7_35_bq.merge(df_sultra_kel_rtv_weighted_calculated_7_51_bq, on='kode_kelurahan', how='left', suffixes=('', '_del'))
df_merge_35_51 = df_merge_35_51.drop(df_merge_35_51.columns[df_merge_35_51.columns.str.endswith('_del')], axis=1)
df_merge_35_51['kode_kecamatan'] = df_merge_35_51['kode_kelurahan'].str[:8]
df_merge_35_51.columns





Index(['kode_kelurahan', 'kode_kabupaten', 'kelurahan', 'kabupaten',
       'Total Suara DPT', 'Target Suara 35%', 'Target Orang yang Ditemui 35%',
       'Target Visit Rumah 35%', 'Target D2D Perhari 35%',
       'Kebutuhan Relawan 35%', 'Target Suara 51%',
       'Target Orang yang Ditemui 51%', 'Target Visit Rumah 51%',
       'Target D2D Perhari 51%', 'Kebutuhan Relawan 51%', 'kode_kecamatan'],
      dtype='object')

In [14]:
gdf_merge_kel, gdf_merge_kec, gdf_merge_kab_kota = functions.join_geojson(df_merge_35_51)

kel_list = list(gdf_merge_kel.NAMA_KEL_DESA.unique())
kec_list = list(gdf_merge_kel.NAMA_KECAMATAN.unique())
kab_list = list(gdf_merge_kel.NAMA_KAB_KOTA.unique())

kel_list.sort()
kec_list.sort()
kab_list.sort()

plot_list = ['Total Suara DPT', 'Target Suara 35%', 'Target Orang yang Ditemui 35%',
                'Target Visit Rumah 35%', 'Target D2D Perhari 35%', 'Kebutuhan Relawan 35%',
                'Target Suara 51%', 'Target Orang yang Ditemui 51%', 'Target Visit Rumah 51%',
                'Target D2D Perhari 51%', 'Kebutuhan Relawan 51%']

# plotting_numeric.py
import matplotlib.pyplot as plt
import contextily as cx
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import logging

import matplotlib
matplotlib.use('Agg')  # Use the 'Agg' backend for non-GUI rendering

In [5]:
gdf_merge_kel

,KODE_KEL_DESA,KODE_KEC,KODE_KAB_KOTA,KODE_PROVINSI,NAMA_KEL_DESA,NAMA_KECAMATAN,NAMA_KAB_KOTA,NAMA_PROVINSI,LUAS_HA,LUAS_SQKM,...,Target Orang yang Ditemui 35%,Target Visit Rumah 35%,Target D2D Perhari 35%,Kebutuhan Relawan 35%,Target Suara 51%,Target Orang yang Ditemui 51%,Target Visit Rumah 51%,Target D2D Perhari 51%,Kebutuhan Relawan 51%,kode_kecamatan
0,74.13.06.2004,74.13.06,74.13,74,Abadi Jaya,Maginti,Muna Barat,Sulawesi Tenggara,787.697137,7.876971,...,949.111033,372.931644,3.729316,1.243105,811.695866,1382.990363,543.414681,5.434147,1.811382,74.13.06
1,74.02.40.2004,74.02.40,74.02,74,Abelisawah,Anggalomoare,Konawe,Sulawesi Tenggara,35.180880,0.351809,...,265.026683,104.136221,1.041362,0.347121,192.041684,386.181738,151.741351,1.517414,0.505805,74.02.40
2,74.05.05.2022,74.05.05,74.05,74,Abenggi,Landono,Konawe Selatan,Sulawesi Tenggara,383.293958,3.832940,...,591.451214,232.397334,2.323973,0.774658,485.447609,861.828912,338.636115,3.386361,1.128787,74.05.05
3,74.09.05.2006,74.09.05,74.09,74,Abola,Lasolo,Konawe Utara,Sulawesi Tenggara,290.958842,2.909588,...,348.320783,136.864748,1.368647,0.456216,267.148106,507.553142,199.431490,1.994315,0.664772,74.09.05
4,74.02.25.2004,74.02.25,74.02,74,Abuhu,Meluhu,Konawe,Sulawesi Tenggara,182.837412,1.828374,...,444.866218,174.800086,1.748001,0.582667,340.158886,648.233632,254.708696,2.547087,0.849029,74.02.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2279,74.07.07.1011,74.07.07,74.07,74,Rukuwa,Tomia Timur,Wakatobi,Sulawesi Tenggara,10.443679,0.104437,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2280,74.13.06.2006,74.13.06,74.13,74,Pasipadanga,Maginti,Muna Barat,Sulawesi Tenggara,28.353911,0.283539,...,340.433084,133.765456,1.337655,0.445885,307.566258,496.059637,194.915378,1.949154,0.649718,74.13.06
2281,74.10.01.2024,74.10.01,74.10,74,Banu-Banua Jaya,Kulisusu,Buton Utara,Sulawesi Tenggara,5.024187,0.050242,...,251.775349,98.929410,0.989294,0.329765,178.465023,366.872651,144.154283,1.441543,0.480514,74.10.01
2282,74.07.02.2018,74.07.02,74.07,74,Mantigola Makmur,Kaledupa,Wakatobi,Sulawesi Tenggara,219.472807,2.194728,...,289.320796,113.682042,1.136820,0.378940,206.255515,421.581731,165.650975,1.656510,0.552170,74.07.02


# Kabupaten

In [6]:
# # plotting_numeric.py
# import matplotlib.pyplot as plt
# import contextily as cx
# from mpl_toolkits.axes_grid1 import make_axes_locatable
# import os
# import logging

# import matplotlib
# matplotlib.use('Agg')  # Use the 'Agg' backend for non-GUI rendering

# def map_numeric_kabupaten(data, col):
#     fig, ax = plt.subplots(figsize = (10,15))
#     divider = make_axes_locatable(ax)
#     cax = divider.append_axes("bottom", size="3%", pad=0.1)

#     map = data.plot(ax=ax,
#         column=col,
#         edgecolor = 'k',
#         cmap = 'Reds',
#         linewidth = 0.5,
#         legend= True,  
#         alpha = 0.7,
#         cax = cax,
#         categorical = False,
#         legend_kwds={"orientation": "horizontal"},
#             missing_kwds={
#             "color": "black",
#             "edgecolor": "red",
#             # "hatch": "///",
#             "label": "Missing values",
#         }, 
#         )


#     map = cx.add_basemap(ax, source=cx.providers.CartoDB.PositronNoLabels, crs=data.crs, zoom=10)
#     map = ax.set_axis_off()

#     title_font = {'fontsize': 20, 'fontweight': 'bold', 'fontname': 'Helvetica'}  # Adjust the font properties as needed
#     map = ax.set_title(f"Peta {col}".upper(), pad=2, loc='left', fontsize=title_font['fontsize'],  fontdict=title_font)

#     # Add labels to each geometry
#     for idx, row in data.iterrows():
#         # Get the centroid for placing the label
#         centroid = row.geometry.centroid
#         label = f"{row['NAMA_KAB_KOTA']}\n{row[col]:,.0f}"
#         # Use `ax.text` to place the label
#         # Adjust horizontal and vertical alignment and offset if needed
#         ax.text(
#             centroid.x, centroid.y, str(label),
#             horizontalalignment='center',
#             verticalalignment='center',
#             fontsize=8,  # Adjust the fontsize as needed
#             color='black',  # Choose a color that contrasts with your map
#             bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=0.5)  # Optional: add a background to the text
#         )

#     # Save the plot to a file
#     filename = f'new_plot/kab_kota/map_of_{col}.png'  # Construct the filename
#     plt.savefig(filename, dpi=300, bbox_inches='tight')  # Save the figure

#     plt.close(fig)  # Close the figure to free memory

# gdf_merge_kel, gdf_merge_kec, gdf_merge_kab_kota = functions.join_geojson(df_merge_35_51)

# plot_list = ['Total Suara DPT', 'Target Suara 35%', 'Target Orang yang Ditemui 35%',
#                 'Target Visit Rumah 35%', 'Target D2D Perhari 35%', 'Kebutuhan Relawan 35%',
#                 'Target Suara 51%', 'Target Orang yang Ditemui 51%', 'Target Visit Rumah 51%',
#                 'Target D2D Perhari 51%', 'Kebutuhan Relawan 51%']
    
# for item in plot_list:
#     map_numeric_kabupaten(gdf_merge_kab_kota, item)
#     print(f'{item} is done plotting')

# Kecamatan

In [7]:
# def map_numeric_kecamatan(data, col):
#     fig, ax = plt.subplots(figsize = (10,15))
#     divider = make_axes_locatable(ax)
#     cax = divider.append_axes("bottom", size="3%", pad=0.1)

#     map = data.plot(ax=ax,
#         column=col,
#         edgecolor = 'k',
#         cmap = 'Reds',
#         linewidth = 0.5,
#         legend= True,  
#         alpha = 0.7,
#         cax = cax,
#         categorical = False,
#         legend_kwds={"orientation": "horizontal"},
#             missing_kwds={
#             "color": "black",
#             "edgecolor": "red",
#             # "hatch": "///",
#             "label": "Missing values",
#         }, 
#         )


#     map = cx.add_basemap(ax, source=cx.providers.CartoDB.PositronNoLabels, crs=data.crs, zoom=12)
#     map = ax.set_axis_off()

#     title_font = {'fontsize': 20, 'fontweight': 'bold', 'fontname': 'Helvetica'}  # Adjust the font properties as needed
#     map = ax.set_title(f"Peta {col} di {kabupaten}".upper(), pad=2, loc='left', fontsize=title_font['fontsize'],  fontdict=title_font)

#     # Add labels to each geometry
#     for idx, row in data.iterrows():
#         # Get the centroid for placing the label
#         centroid = row.geometry.centroid
#         label = f"{row['NAMA_KECAMATAN']}\n{row[col]:,.2f}"
#         # Use `ax.text` to place the label
#         # Adjust horizontal and vertical alignment and offset if needed
#         ax.text(
#             centroid.x, centroid.y, str(label),
#             horizontalalignment='center',
#             verticalalignment='center',
#             fontsize=6,  # Adjust the fontsize as needed
#             color='black',  # Choose a color that contrasts with your map
#             bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=0.5)  # Optional: add a background to the text
#         )

#      # Ensure the directory exists
#     directory = f'new_plot/kab_kota/{kabupaten}'
#     if not os.path.exists(directory):
#         os.makedirs(directory)

#     # Save the plot to a file
#     filename = f'{directory}/map_of_{col}.png'  # Construct the filename
#     plt.savefig(filename, dpi=300, bbox_inches='tight')  # Save the figure

#     plt.close(fig)  # Close the figure to free memory

# for kabupaten in kab_list:
#     for item in plot_list:
#         map_numeric_kecamatan(gdf_merge_kec[gdf_merge_kec.NAMA_KAB_KOTA == kabupaten], item)
#         print(f'{kabupaten}_{item} is done plotting')

# Kelurahan

In [11]:
kab_list.sort()
kab_list

['Bombana',
 'Buton',
 'Buton Selatan',
 'Buton Tengah',
 'Buton Utara',
 'Kolaka',
 'Kolaka Timur',
 'Kolaka Utara',
 'Konawe',
 'Konawe Kepulauan',
 'Konawe Selatan',
 'Konawe Utara',
 'Kota Bau Bau',
 'Kota Kendari',
 'Muna',
 'Muna Barat',
 'Wakatobi']

In [16]:
def map_numeric_kelurahan(data, col):
    fig, ax = plt.subplots(figsize = (10,15))
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("bottom", size="3%", pad=0.1)

    map = data.plot(ax=ax,
        column=col,
        edgecolor = 'k',
        cmap = 'Reds',
        linewidth = 0.5,
        legend= True,  
        alpha = 0.7,
        cax = cax,
        categorical = False,
        legend_kwds={"orientation": "horizontal"},
            missing_kwds={
            "color": "black",
            "edgecolor": "red",
            # "hatch": "///",
            "label": "Missing values",
        }, 
        )


    map = cx.add_basemap(ax, source=cx.providers.CartoDB.PositronNoLabels, crs=data.crs, zoom=15)
    map = ax.set_axis_off()

    title_font = {'fontsize': 20, 'fontweight': 'bold', 'fontname': 'Helvetica'}  # Adjust the font properties as needed
    map = ax.set_title(f"Peta {col} di {kabupaten}, {kecamatan}".upper(), pad=2, loc='left', fontsize=title_font['fontsize'],  fontdict=title_font)

    # Add labels to each geometry
    for idx, row in data.iterrows():
        # Get the centroid for placing the label
        centroid = row.geometry.centroid
        label = f"{row['NAMA_KEL_DESA']}\n{row[col]:,.2f}"
        # Use `ax.text` to place the label
        # Adjust horizontal and vertical alignment and offset if needed
        ax.text(
            centroid.x, centroid.y, str(label),
            horizontalalignment='center',
            verticalalignment='center',
            fontsize=6,  # Adjust the fontsize as needed
            color='black',  # Choose a color that contrasts with your map
            bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=0.5)  # Optional: add a background to the text
        )

     # Ensure the directory exists
    directory = f'new_plot/kab_kota/{kabupaten}/{kecamatan}'
    if not os.path.exists(directory):
        os.makedirs(directory)

    # Save the plot to a file
    filename = f'{directory}/map_of_{col}.png'  # Construct the filename
    plt.savefig(filename, dpi=300, bbox_inches='tight')  # Save the figure

    plt.close(fig)  # Close the figure to free memory

for kabupaten in kab_list:
    kab_gdf = gdf_merge_kel[gdf_merge_kel.NAMA_KAB_KOTA == kabupaten]
    kec_list = (kab_gdf.NAMA_KECAMATAN.unique())
    for kecamatan in kec_list:
        for item in plot_list:
            map_numeric_kelurahan(kab_gdf[kab_gdf.NAMA_KECAMATAN == kecamatan], item)
            print(f'{kabupaten}_{kecamatan}_{item} is done plotting')

Bombana_Poleang Selatan_Total Suara DPT is done plotting
Bombana_Poleang Selatan_Target Suara 35% is done plotting
Bombana_Poleang Selatan_Target Orang yang Ditemui 35% is done plotting
Bombana_Poleang Selatan_Target Visit Rumah 35% is done plotting
Bombana_Poleang Selatan_Target D2D Perhari 35% is done plotting
Bombana_Poleang Selatan_Kebutuhan Relawan 35% is done plotting
Bombana_Poleang Selatan_Target Suara 51% is done plotting
Bombana_Poleang Selatan_Target Orang yang Ditemui 51% is done plotting
Bombana_Poleang Selatan_Target Visit Rumah 51% is done plotting
Bombana_Poleang Selatan_Target D2D Perhari 51% is done plotting
Bombana_Poleang Selatan_Kebutuhan Relawan 51% is done plotting
Bombana_Poleang Barat_Total Suara DPT is done plotting
Bombana_Poleang Barat_Target Suara 35% is done plotting
Bombana_Poleang Barat_Target Orang yang Ditemui 35% is done plotting
Bombana_Poleang Barat_Target Visit Rumah 35% is done plotting
Bombana_Poleang Barat_Target D2D Perhari 35% is done plotting

KeyboardInterrupt: 

In [ ]:
class NestedMapPlotter:
    def __init__(self, plot_list, gdf_merge_kel, gdf_merge_kec, gdf_merge_kab_kota, logger):
        self.plot_list = plot_list
        self.gdf_merge_kel = gdf_merge_kel
        self.gdf_merge_kec = gdf_merge_kec
        self.gdf_merge_kab_kota = gdf_merge_kab_kota
        self.kab_list = list(gdf_merge_kel.NAMA_KAB_KOTA.unique())
        self.worker_queue = queue.Queue()
        self.logger = logger

    def setup_workers(self, max_workers):
        for i in range(1, max_workers + 1):
            self.worker_queue.put(i)

    def worker_wrapper(self, func, *args):
        worker_id = self.worker_queue.get()
        self.logger.info(f'Worker-{worker_id} started')
        try:
            func(*args, worker_id)
        except Exception as e:
            self.logger.error(f'Worker-{worker_id} encountered an error: {e}')
        finally:
            self.worker_queue.put(worker_id)
            self.logger.info(f'Worker-{worker_id} finished')

    def process_kelurahan(self, kabupaten, worker_id):
        self.logger.info(f'Worker-{worker_id}: Processing kelurahan for {kabupaten}')
        kab_gdf_kel = self.gdf_merge_kel[self.gdf_merge_kel.NAMA_KAB_KOTA == kabupaten]
        kec_list = kab_gdf_kel.NAMA_KECAMATAN.unique()
        for kecamatan in kec_list:
            for item in self.plot_list:
                filename = f'new_plot/kab_kota/{kabupaten}/{kecamatan}/map_of_{item}.png'
                if os.path.exists(filename):
                    self.logger.info(f'Worker-{worker_id}: {kabupaten}_{kecamatan}_{item} already exists, skipping.')
                    continue
                plotting_numeric.map_numeric(kab_gdf_kel[kab_gdf_kel.NAMA_KECAMATAN == kecamatan], item, 'kelurahan', kabupaten, kecamatan, worker_id)
                self.logger.info(f'Worker-{worker_id}: {kabupaten}_{kecamatan}_{item} is done plotting')

    def process_kecamatan(self, kabupaten, worker_id):
        self.logger.info(f'Worker-{worker_id}: Processing kecamatan for {kabupaten}')
        for item in self.plot_list:
            filename = f'new_plot/kab_kota/{kabupaten}/map_of_{item}.png'
            if os.path.exists(filename):
                self.logger.info(f'Worker-{worker_id}: {kabupaten}_{item} already exists, skipping.')
                continue
            plotting_numeric.map_numeric(self.gdf_merge_kec[self.gdf_merge_kec.NAMA_KAB_KOTA == kabupaten], item, 'kecamatan', kabupaten, worker_id=worker_id)
            self.logger.info(f'Worker-{worker_id}: {kabupaten}_{item} is done plotting')

    def process_kabupaten(self, item, worker_id):
        self.logger.info(f'Worker-{worker_id}: Processing kabupaten for {item}')
        filename = f'new_plot/kab_kota/map_of_{item}.png'
        if os.path.exists(filename):
            self.logger.info(f'Worker-{worker_id}: {item} already exists, skipping.')
            return
        plotting_numeric.map_numeric(self.gdf_merge_kab_kota, item, 'kabupaten', worker_id=worker_id)
        self.logger.info(f'Worker-{worker_id}: {item} is done plotting')

    def run(self, max_workers):
        self.setup_workers(max_workers)
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for kabupaten in self.kab_list:
                futures.append(executor.submit(self.worker_wrapper, self.process_kelurahan, kabupaten))
                futures.append(executor.submit(self.worker_wrapper, self.process_kecamatan, kabupaten))

            for item in self.plot_list:
                futures.append(executor.submit(self.worker_wrapper, self.process_kabupaten, item))
            
            concurrent.futures.wait(futures)

# Example usage
if __name__ == '__main__':
    plot_list = ['Total Suara DPT', 'Target Suara 35%', 'Target Orang yang Ditemui 35%',
                 'Target Visit Rumah 35%', 'Target D2D Perhari 35%', 'Kebutuhan Relawan 35%',
                 'Target Suara 51%', 'Target Orang yang Ditemui 51%', 'Target Visit Rumah 51%',
                 'Target D2D Perhari 51%', 'Kebutuhan Relawan 51%']
    
    gdf_merge_kel, gdf_merge_kec, gdf_merge_kab_kota = functions.join_geojson(df_merge_35_51)

    logger = setup_logger('MapPlotter')
    plotter = NestedMapPlotter(plot_list, gdf_merge_kel, gdf_merge_kec, gdf_merge_kab_kota, logger)
    plotter.run(max_workers=40)



2024-07-06 14:58:23,255 - MapPlotter - INFO - Worker-1 started
2024-07-06 14:58:23,256 - MapPlotter - INFO - Worker-2 started
2024-07-06 14:58:23,256 - MapPlotter - INFO - Worker-3 started
2024-07-06 14:58:23,259 - MapPlotter - INFO - Worker-4 started
2024-07-06 14:58:23,259 - MapPlotter - INFO - Worker-5 started
2024-07-06 14:58:23,259 - MapPlotter - INFO - Worker-1: Processing kelurahan for Muna Barat
2024-07-06 14:58:23,259 - MapPlotter - INFO - Worker-6 started
2024-07-06 14:58:23,260 - MapPlotter - INFO - Worker-7 started
2024-07-06 14:58:23,260 - MapPlotter - INFO - Worker-8 started
2024-07-06 14:58:23,260 - MapPlotter - INFO - Worker-2: Processing kecamatan for Muna Barat
2024-07-06 14:58:23,260 - MapPlotter - INFO - Worker-9 started
2024-07-06 14:58:23,260 - MapPlotter - INFO - Worker-10 started
2024-07-06 14:58:23,261 - MapPlotter - INFO - Worker-3: Processing kelurahan for Konawe
2024-07-06 14:58:23,261 - MapPlotter - INFO - Worker-11 started
2024-07-06 14:58:23,261 - MapPlot